# Practical 3B — Fashion-MNIST Regularization

## Aim
Study regularization on Fashion-MNIST using a deep multilayer perceptron.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load Fashion-MNIST and prepare splits

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
np.random.seed(42); tf.random.set_seed(42)
(x_all,y_all),(x_test,y_test)=keras.datasets.fashion_mnist.load_data()
x_all=x_all.astype("float32")/255.; x_test=x_test.astype("float32")/255.
x_train,x_val=x_all[5000:],x_all[:5000]; y_train,y_val=y_all[5000:],y_all[:5000]
print(x_train.shape,x_val.shape,x_test.shape)

## 2. Model and gradient utilities

In [ ]:
def make_model(mode="poor"):
    init="random_normal" if mode=="poor" else ("glorot_uniform" if mode=="xavier" else "he_normal")
    model=keras.Sequential([layers.Input((28,28)),layers.Flatten()])
    for _ in range(5):
        model.add(layers.Dense(256,activation="relu",kernel_initializer=init))
    model.add(layers.Dense(10,activation="softmax"))
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

def gradient_norms(model, x, y):
    with tf.GradientTape() as tape:
        pred=model(x,training=True)
        loss=keras.losses.sparse_categorical_crossentropy(y,pred)
        loss=tf.reduce_mean(loss)
    grads=tape.gradient(loss,model.trainable_variables)
    return [float(tf.norm(g)) for g in grads if g is not None]

## 3. Compare baseline, dropout, L1 and L2 regularization

In [ ]:
def regularized_model(kind):
    reg=None if kind in ["baseline","dropout"] else regularizers.l1(1e-5) if kind=="l1" else regularizers.l2(1e-4)
    model=keras.Sequential([layers.Input((28,28)),layers.Flatten()])
    for _ in range(3):
        model.add(layers.Dense(256,activation="relu",kernel_regularizer=reg))
        if kind=="dropout": model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10,activation="softmax"))
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model
runs={}
for kind in ["baseline","dropout","l1","l2"]:
    m=regularized_model(kind); h=m.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=20,batch_size=256,verbose=0)
    runs[kind]=(m,h); print(kind,"validation accuracy",h.history["val_accuracy"][-1])

In [ ]:
for k,(m,h) in runs.items(): plt.plot(h.history["val_loss"],label=k)
plt.title("Validation loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

**Conclusion:** Compare validation accuracy and loss. Dropout randomly disables units during training; L1 encourages sparse weights; L2 penalizes large weights. Regularization strength affects results.